# 01 交通数据质量审计

你是道路监测数据的分析助理。建模小组准备直接使用收到的CSV，请提交一份数据接收审计报告，决定哪些记录可用、如何组织，以及哪些问题仍需向数据提供方核实。

建议工作量：4—6小时。本工作本是项目起点，默认程序的输出不是完整作业答案。

## 最终成果
- 数据字典与质量审计表
- 去重前后统计对照CSV
- 可执行数据处理规则
- 两页数据接收结论


## 数据与范围

[UCI Metro Interstate Traffic Volume](https://archive.ics.uci.edu/dataset/492/metro+interstate+traffic+volume)

CC BY 4.0，John Hogue (2019)

保留原始下载哈希。工作台按小时合并相同交通量，保存原行数权重用于还原重复记录对均值的影响。

- 记录行数不等于不同观测小时数。
- 零交通量不能直接判为缺失。
- 时间标签的缺口与字段中的空值不同。
- 原数据使用本地时间标签，本项目不恢复夏令时。


In [ ]:
from pathlib import Path
import sys, json
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "python" / "analyze.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the project-kit folder or its notebooks folder")
sys.path.insert(0, str(ROOT / "python"))
from analyze import run
OUTPUT = ROOT / "outputs" / "student"
OUTPUT.mkdir(parents=True, exist_ok=True)


## 参考分析与口径检查

先运行一次，解释每个指标的分母和单位。打开源码确认数据筛选规则。预测项目这里读取冻结模型的结果，不重新训练。


In [ ]:
project = "audit"
config = {
  "year": "2018",
  "method": "unique"
}
result = run(project, config, ROOT / "data")
print(json.dumps(result["metrics"], ensure_ascii=False, indent=2))
print("Source SHA-256:", result["sourceSha"])


## 对照实验

以下配置提供一个可运行起点。说明每次只改变了什么，以及还存在哪些混杂条件。增加你自己的对照，不只重复默认结果。


In [ ]:
comparisons = [
  {
    "year": "2018",
    "method": "unique"
  },
  {
    "year": "2018",
    "method": "raw"
  },
  {
    "year": "2017",
    "method": "unique"
  }
]
experiments = []
for change in comparisons:
    trial = run(project, {**config, **change}, ROOT / "data")
    experiments.append({"project": project, "config": trial["config"], "metrics": trial["metrics"], "sourceSha": trial["sourceSha"]})
    print(json.dumps(experiments[-1], ensure_ascii=False))
(OUTPUT / (project + "-comparison.json")).write_text(json.dumps(experiments, ensure_ascii=False, indent=2), encoding="utf-8")


## 01 数据契约

**明确一行数据应该代表什么**

写明检测点、方向、时间单位、交通量单位，并区分原始行与观测小时。

阶段成果：数据字典：至少说明时间、交通量、天气、原行数权重。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 02 质量核查

**建立证据清单**

按年份检查原始行、唯一小时、重复行与时间缺口。为什么“无字段缺失”仍不够？

阶段成果：质量表中必须给出分母、统计范围与缺口处理原则。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 03 处理对照

**量化重复记录的影响**

分别保留原始行权重和按小时去重，比较24小时均值。挑出变化较大的时段解释原因。

阶段成果：至少保存两次不同处理配置的结果，不只贴一张图。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 04 接收结论

**提出可执行处理规则**

哪些字段用于分组，哪些记录保留或排除？列出无法凭这份文件确认的问题。

阶段成果：提交处理规则、限制清单和接收结论，避免删除一切异常值。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 深入分析

- 将审计规则迁移到自己的交通数据，并比较字段空值和时间缺口。
- 检查不同天气标签是否导致同一小时多行，解释为什么不能直接求和。

修改 python/analyze.py 或 python/prepare_data.py 前，先复制为自己的版本并保留数据与参数来源。


## 提交前自查

- [ ] 质量计数与下载CSV一致。
- [ ] 说明去重依据，未将时间缺口填成0。
- [ ] 至少2次对照记录，包含一项不确定性。

报告应包含研究问题、方法对照、发现、局限、源数据哈希和复现命令。请附代码、配置、结果CSV。阶段文字与实验次数不自动换算成绩。
